In [5]:
import requests
import pandas as pd
from datetime import datetime, timezone
import time
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.colors import BoundaryNorm
from matplotlib.cm import get_cmap
from matplotlib.patches import Patch
import ast
import folium
from folium.plugins import MarkerCluster
import reverse_geocoder as rg
import re
import pycountry
import os
import numpy as np
import geopandas as gpd
import fiona
import sys
from shapely.geometry import Point
from sklearn.cluster import DBSCAN
import ruptures as rpt
from haversine import haversine
import functions as own
from timezonefinder import TimezoneFinder
import zoneinfo
from scipy.stats import gaussian_kde
from scipy.stats import chisquare, kruskal, mannwhitneyu, spearmanr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from libpysal.weights import lat2W
from libpysal.weights import Queen
from libpysal.weights import DistanceBand
from esda.moran import Moran_Local, Moran
from shapely.geometry import box
import statsmodels.api as sm
from statsmodels.tsa.seasonal import STL
import importlib

In [20]:
importlib.reload(own)

<module 'functions' from 'c:\\Studium\\X_Masterarbeit\\Data\\Master_Thesis\\Code\\functions.py'>

A barchart is created that depicts the hour of the day when observations are made, is created.

In [9]:
df = pd.read_csv("../CWData_clean7.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])

df["hour"] = df["created_at_local"].dt.hour
counts = df["hour"].value_counts().reindex(range(24), fill_value=0).sort_index()

ax = counts.plot(kind="bar", color="teal", figsize=(16,9), logy = False)

plt.title("Total Observations per Hour of Day (Local Time)", fontsize=20)
plt.xlabel("Hour of Day", fontsize=17)
plt.ylabel("Number of Observations", fontsize=17)
plt.xticks(rotation=90)

#for p in ax.patches: # absolute counts above bars
#    ax.annotate(
#        f'{int(p.get_height())}', 
#        (p.get_x() + p.get_width() / 2., p.get_height()), 
#        ha='center', 
#        va='bottom', 
#        fontsize=10,
#        xytext=(0, 5),
#        textcoords='offset points',
#        rotation=90
#    )

ax.grid(axis="y", linestyle="--", alpha=0.5)
ax.set_xticklabels([f"{h:02d}:00-{(h+1)%24:02d}:00" for h in range(24)], rotation=45, ha="right", fontsize=12)
ax.set_axisbelow(True)
ax.set_ylim(0,6500)

plt.savefig(f"../Products/hour_of_day.png", dpi=300, bbox_inches="tight")
plt.close()

C:\Users\yanni\AppData\Local\Temp\ipykernel_14684\4183465156.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 

Chi-square goodness-of-fit test

In [5]:
observed = df["hour"].value_counts().sort_index()
chi2, p = chisquare(observed)
expected = observed.sum() / len(observed) # uniform distribution
residuals = (observed - expected) / np.sqrt(expected)
print(f"{chi2}, {p}, \n {residuals}")

48096.0483621203, 0.0, 
 hour
0    -49.723428
1    -51.897371
2    -52.332160
3    -52.577910
4    -52.766948
5    -51.670525
6    -44.071177
7    -16.566074
8     50.448076
9     33.415707
10    30.050821
11    43.548171
12    50.277941
13    42.792017
14    46.213614
15    62.300790
16    63.945425
17    58.406597
18    26.043205
19    -2.425994
20   -22.766537
21   -28.229750
22   -37.795099
23   -44.619389
Name: count, dtype: float64


Then, a barchart with the month of the year

In [13]:
df = pd.read_csv("../CWData_clean7.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])

df["month"] = df["created_at_local"].dt.month
df["year"] = df["created_at_local"].dt.year
total_obs = df["month"].value_counts().reindex(range(1,13), fill_value=0).sort_index()
years_per_month = df.groupby("month")["year"].nunique()
counts = (total_obs / years_per_month).round(0).astype(int) # because February, March and April are more common than the rest

ax = counts.plot(kind="bar", color="teal", figsize=(16,9), logy = False)

plt.title("Average Observations per Month of Year", fontsize=20)
plt.xlabel("Month of Year", fontsize=17)
plt.ylabel("Number of Observations", fontsize=17)
plt.xticks(rotation=90)

for p in ax.patches: # absolute counts above bars
    ax.annotate(
        f'{int(p.get_height())}', 
        (p.get_x() + p.get_width() / 2., p.get_height()), 
        ha='center', 
        va='bottom', 
        fontsize=10,
        xytext=(0, 5),
        textcoords='offset points',
        rotation=90
    )

ax.grid(axis="y", linestyle="--", alpha=0.5)
ax.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], rotation=45, ha="right", fontsize=12)
ax.set_axisbelow(True)
ax.set_ylim(0,1000)

plt.savefig(f"../Products/month_of_year.png", dpi=300, bbox_inches="tight")
plt.close()

C:\Users\yanni\AppData\Local\Temp\ipykernel_14684\3863024384.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 

Chi-square goodness-of-fit test

In [11]:
observed = df["month"].value_counts().sort_index()
chi2, p = chisquare(observed)
expected = observed.sum() / len(observed) # uniform distribution
residuals = (observed - expected) / np.sqrt(expected)
print(f"{chi2}, {p}, \n {residuals}")

2017.9362715902323, 0.0, 
 month
1    -19.297551
2    -11.157024
3      8.906905
4     16.686523
5     24.385938
6     11.526845
7      1.969411
8      0.966883
9     -1.078275
10    -4.286364
11   -13.375952
12   -15.247338
Name: count, dtype: float64


Then, a barchart with the year

In [25]:
df = pd.read_csv("../CWData_clean7.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])
df["month"] = df["created_at_local"].dt.month
df["year"] = df["created_at_local"].dt.year
counts = df["year"].value_counts().reindex(range(2017,2027), fill_value=0).sort_index()

ax = counts.plot(kind="bar", color="teal", figsize=(16,9), logy = False)

# extrapolation for 2026 (after April)
full_years = range(2018, 2026) # 2018 to 2025
april_fracs = []
for y in full_years:
    total = (df["year"] == y).sum()
    until_april = ((df["year"] == y) & (df["month"] <= 4)).sum()
    april_fracs.append(until_april / total)

avg_april_frac = np.mean(april_fracs)

obs_2026_actual = (df["year"] == 2026).sum()
obs_2026_extrap = int(obs_2026_actual / avg_april_frac) - obs_2026_actual
print(obs_2026_actual, obs_2026_extrap, avg_april_frac)

bars = ax.patches
last_bar = bars[-1]  # 2026-Bar
ax.bar(
    last_bar.get_x() + last_bar.get_width() / 2,
    obs_2026_extrap,
    bottom=obs_2026_actual,
    width=last_bar.get_width(),
    color="#90E4C1",
    hatch="//",
    edgecolor="teal",
    label=f"2026 extrapolated (÷{avg_april_frac:.0%} Apr avg)"
)

plt.title("Total Observations per Year", fontsize=20)
plt.xlabel("Year", fontsize=17)
plt.ylabel("Number of Observations", fontsize=17)
plt.xticks(rotation=90)

#for p in ax.patches: # absolute counts above bars
#    ax.annotate(
#        f'{int(p.get_height())}', 
#        (p.get_x() + p.get_width() / 2., p.get_height()), 
#        ha='center', 
#        va='bottom', 
#        fontsize=10,
#        xytext=(0, 5),
#        textcoords='offset points',
#        rotation=90
#    )

ax.grid(axis="y", linestyle="--", alpha=0.5)
ax.set_xticklabels(["2017 (from Feb onwards)","2018","2019","2020","2021","2022","2023","2024","2025","2026 (extrapolated)"], rotation=45, ha="right", fontsize=12)
ax.set_axisbelow(True)
ax.set_ylim(0,13000)

plt.savefig(f"../Products/year.png", dpi=300, bbox_inches="tight")
plt.close()

C:\Users\yanni\AppData\Local\Temp\ipykernel_14684\3796041459.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 

2481 6074 0.28997957679063646


Chi-square goodness-of-fit test + quadratic regression

In [26]:
observed = df[(df["year"] > 2017) & (df["year"] < 2026)]["year"].value_counts().sort_index()
chi2, p = chisquare(observed)
expected = observed.sum() / len(observed) # uniform distribution
residuals = (observed - expected) / np.sqrt(expected)
print(f"{chi2}, {p}, \n {residuals}")

yearly = df.groupby("year").size().sort_index()

x = yearly.index.values # x (centered)
x_c = x - x.mean()

X = np.column_stack([x_c, x_c**2]) # design matrix
X = sm.add_constant(X)

y = yearly.values

model = sm.OLS(y, X).fit()
print(model.summary())

6817.286334530689, 0.0, 
 year
2018   -59.675055
2019    -9.855955
2020    -1.729752
2021    42.075734
2022    13.505481
2023    15.785736
2024    21.788172
2025   -21.894360
Name: count, dtype: float64
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.919
Model:                            OLS   Adj. R-squared:                  0.896
Method:                 Least Squares   F-statistic:                     39.66
Date:                Wed, 20 May 2026   Prob (F-statistic):           0.000152
Time:                        09:57:27   Log-Likelihood:                -83.236
No. Observations:                  10   AIC:                             172.5
Df Residuals:                       7   BIC:                             173.4
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
       

And a chart that shows the annual ratio between new and existing users

In [30]:
df = pd.read_csv("../CWData_clean7.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])
df["year"] = df["created_at_local"].dt.year

all_years = sorted(df["year"].unique())

seen_users = set()
results = []

for year in all_years:
    year_users = set(df[df["year"] == year]["created_by"].dropna().unique())
    
    new_users = year_users - seen_users
    returning_users = year_users & seen_users
    
    results.append({
        "year": year,
        "new_users": len(new_users),
        "returning_users": len(returning_users),
        "total_users": len(year_users)
    })
    
    seen_users.update(year_users)

result_df = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(16, 9))

ax.bar(
    result_df["year"],
    result_df["returning_users"],
    label="Returning Users",
    color="teal",
    width=0.6
)
ax.bar(
    result_df["year"],
    result_df["new_users"],
    bottom=result_df["returning_users"],
    label="New Users",
    color="lightblue",
    width=0.6
)

#for _, row in result_df.iterrows():
#    ratio = row["new_users"] / row["total_users"] * 100
#    ax.annotate(
#        f'{ratio:.0f}% new',
#        (row["year"], row["total_users"]),
#        ha='center',
#        va='bottom',
#        fontsize=9,
#        xytext=(0, 5),
#        textcoords='offset points'
#    )

ax.set_xticks(result_df["year"])
ax.set_xticklabels(result_df["year"], rotation=45, ha="right", fontsize=12)
ax.set_title("Annual New vs. Existing Users", fontsize=20)
ax.set_xlabel("Year", fontsize=17)
ax.set_ylabel("Number of Users", fontsize=17)
ax.legend(fontsize=12)
ax.grid(axis="y", linestyle="--", alpha=0.5)
ax.set_axisbelow(True)
ax.set_ylim(0,700)

plt.savefig("../Products/new_vs_existing_users_annual.png", dpi=300, bbox_inches="tight")
plt.close()

C:\Users\yanni\AppData\Local\Temp\ipykernel_14684\2263137817.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 

And also the monthly ratio between new and existing users

In [2]:
df = pd.read_csv("../CWData_clean7.csv")
df["year_month"] = pd.to_datetime(df["year_month"]).dt.to_period("M")

all_months = sorted(df["year_month"].unique())

seen_users = set()
results = []

for month in all_months:
    month_users = set(df[df["year_month"] == month]["spotted_by"].dropna().unique())
    
    new_users = month_users - seen_users
    existing_users = month_users & seen_users
    
    results.append({
        "year_month": month,
        "new_users": len(new_users),
        "existing_users": len(existing_users),
        "total_users": len(month_users)
    })
    
    seen_users.update(month_users)

result_df = pd.DataFrame(results)
result_df["year_month_dt"] = result_df["year_month"].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(16, 9))

ax.bar(
    result_df["year_month_dt"],
    result_df["existing_users"],
    label="Returning Users",
    color="teal",
    width=20
)
ax.bar(
    result_df["year_month_dt"],
    result_df["new_users"],
    bottom=result_df["existing_users"],
    label="New Users",
    color="lightblue",
    width=20
)

for _, row in result_df.iterrows():
    ratio = row["new_users"] / row["total_users"] * 100
    ax.annotate(
        f'{ratio:.0f}% new',
        (row["year_month_dt"], row["total_users"]),
        ha='center',
        va='bottom',
        fontsize=5.5,
        rotation=90,
        xytext=(0, 2),
        textcoords='offset points'
    )

ax.set_title("Monthly New vs. Existing Users", fontsize=20)
ax.set_xlabel("Month", fontsize=17)
ax.set_ylabel("Number of Users", fontsize=17)
ax.tick_params(axis="x", labelsize=12)
ax.tick_params(axis="y", labelsize=12)
ax.legend(fontsize=12)
ax.grid(axis="y", linestyle="--", alpha=0.5)
ax.set_axisbelow(True)
ax.set_ylim(0, 200)

plt.savefig("../Products/new_vs_existing_users_monthly.png", dpi=300, bbox_inches="tight")
plt.close()

C:\Users\yanni\AppData\Local\Temp\ipykernel_39640\2298316351.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 

In [3]:
pd.set_option("display.max_rows", None)
result_df

,year_month,new_users,existing_users,total_users,year_month_dt
0,2017-02,2,0,2,2017-02-01
1,2017-03,8,2,10,2017-03-01
2,2017-04,3,4,7,2017-04-01
3,2017-05,15,8,23,2017-05-01
4,2017-06,7,13,20,2017-06-01
5,2017-07,11,9,20,2017-07-01
6,2017-08,4,10,14,2017-08-01
7,2017-09,10,16,26,2017-09-01
8,2017-10,2,15,17,2017-10-01
9,2017-11,3,9,12,2017-11-01


STL decomposition of these data

In [21]:
own.STL_decomposition(result_df, "total_users", "Total Users")
own.STL_decomposition(result_df, "new_users", "New Users")
own.STL_decomposition(result_df, "existing_users", "Existing Users")

Seasonal strength Total Users: 0.779
Trend strength Total Users:    0.840
Seasonal strength New Users: 0.747
Trend strength New Users:    0.528
Seasonal strength Existing Users: 0.602
Trend strength Existing Users:    0.910


Here, monthly maps of observations are created

In [ ]:
df = pd.read_csv("../CWData_clean7.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])

monthly_counts = (
    df
    .groupby(["year_month", "latitude", "longitude"])
    .size()
    .reset_index(name="count")
)

world = gpd.read_file(f"../ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp", engine="fiona").to_crs("+proj=robin")

for month in monthly_counts["year_month"].unique():
    subset = monthly_counts[monthly_counts["year_month"] == month]
    robin_subset = gpd.GeoDataFrame(
        subset,
        geometry=gpd.points_from_xy(subset["longitude"], subset["latitude"]),
        crs="EPSG:4326"
    ).to_crs("+proj=robin")

    fig, ax = plt.subplots(figsize=(16,8))
    
    # draw world map
    world.plot(ax=ax, color="lightgrey", edgecolor="black", linewidth=0.5)
    
    # plot points
    ax.scatter(
        robin_subset.geometry.x,
        robin_subset.geometry.y,
        s=np.log1p(robin_subset["count"]) * 40, # log for radius
        color="teal",
        edgecolor="black",
        linewidth=0.3,
        alpha=0.7
    )

    legend_counts = [1, 10, 50, 100]
    legend_handles = [
        plt.scatter([], [], s=np.log1p(c) * 40, color="teal", edgecolor="black", linewidth=0.3, alpha=0.7, label=str(c))
        for c in legend_counts
    ]
    legend = ax.legend(
        handles=legend_handles,
        title="Observations",
        loc="lower left",
        frameon=True,
        labelspacing=1.5
    )
    legend.get_title().set_fontsize(12)
    for text in legend.get_texts():
        text.set_fontsize(10)
    
    ax.set_title(f"Observations - {month}")
    ax.set_axis_off()
    
    plt.savefig(f"../Products/Monthly_Activity_Maps/map_{month}.png", dpi=300, bbox_inches="tight")
    plt.close()

C:\Users\yanni\AppData\Local\Temp\ipykernel_3356\122382734.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 42

I'm retrieving the date when the first observation is made, the number of observations and the number of unique users per Country.

In [4]:
df = pd.read_csv("../CWData_clean7.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])

first_countries = (
    df
    .groupby("Country")
    .agg(
        first_date = ("created_at_local","min"),
        total_observations = ("created_at_local","count"),
        unique_users = ("created_by","nunique")
    )
    .reset_index()
)

pd.set_option("display.max_rows", None)

first_countries

C:\Users\yanni\AppData\Local\Temp\ipykernel_3356\1938527830.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 4

,Country,first_date,total_observations,unique_users
0,Afghanistan,2017-07-03 12:47:55,1,1
1,Albania,2019-10-17 13:44:38,1,1
2,Argentina,2018-01-13 10:41:01,64,9
3,Australia,2017-12-12 17:38:56,357,14
4,Austria,2017-03-04 14:28:56,6094,95
5,Bangladesh,2023-05-25 13:25:55,4,2
6,Belgium,2017-08-03 21:09:49,12,7
7,Brazil,2020-09-04 13:40:09,139,45
8,Bulgaria,2019-08-13 15:47:52,25,3
9,Cambodia,2022-02-26 11:52:59,4,2


Then, the date of the first observation per country is plotted on a map, this is an overview.

In [ ]:
# uses first_countries

world = gpd.read_file(f"../ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp", engine="fiona")
world = world.rename(columns={"NAME": "Country"})
first_countries["first_date"] = pd.to_datetime(first_countries["first_date"])
first_countries["year"] = first_countries["first_date"].dt.year

bins = [2016, 2017, 2018, 2019, 2020, 2022, 2024, 2026]
labels = ["2017 (Feb-Dec)","2018","2019","2020","2021-2022","2023-2024","2025-2026 (Jan)"]
first_countries["year_bin"] = pd.cut(first_countries["year"], bins=bins, labels=labels)
first_countries["ISO_A3"] = first_countries["Country"].apply(own.get_iso3)

map_df = world.merge(first_countries, left_on="ADM0_A3", right_on="ISO_A3", how="left").to_crs("+proj=robin")

fig, ax = plt.subplots(1, 1, figsize=(16, 8))

cmap = get_cmap("Blues_r", len(labels))
legend_handles = [
    Patch(facecolor=cmap(i / (len(labels) - 1)), edgecolor="black", linewidth=0.5, label=labels[i])
    for i in range(len(labels))
]
legend_handles.append(Patch(facecolor="grey", edgecolor="black", linewidth=0.5, label="No observations"))

map_df.plot(
    column="year_bin",
    cmap="Blues_r",
    linewidth=0.5,
    edgecolor="black",
    missing_kwds={"color": "grey"},
    legend=False,
    categorical=True,
    ax=ax
)

ax.legend(handles=legend_handles, title="Year", title_fontsize=12, loc="lower left")
ax.set_title("First Observation per Country", fontsize=15)
ax.set_axis_off()

plt.savefig(f"../Products/first_obs_per_Country.png", dpi=300, bbox_inches="tight")
plt.close()

Then, monthly maps of first observations are created (same colors as the one above).

In [ ]:
# uses first_countries

world = gpd.read_file(f"../ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp", engine="fiona")
world = world.rename(columns={"NAME": "Country"})
first_countries["ISO_A3"] = first_countries["Country"].apply(own.get_iso3)
first_countries["first_date"] = pd.to_datetime(first_countries["first_date"])

bins = [2016, 2017, 2018, 2019, 2020, 2022, 2024, 2026]
labels = ["2017 (Feb-Dec)","2018","2019","2020","2021-2022","2023-2024","2025-2026 (Jan)"]
first_countries["year_bin"] = pd.cut(first_countries["first_date"].dt.year, bins=bins, labels=labels)

# sort months
all_months = pd.period_range(first_countries["first_date"].min(), first_countries["first_date"].max() + pd.offsets.MonthEnd(2), freq="M")

# generate monthly maps
saved_files = []
for month in all_months:
    current = first_countries[first_countries["first_date"] <= month.to_timestamp()]
    if current.empty:
        continue
    
    map_df = world.merge(current, left_on="ADM0_A3", right_on="ISO_A3", how="left").to_crs("+proj=robin")
    
    fig, ax = plt.subplots(1, 1, figsize=(16, 8))
    
    cmap = get_cmap("Blues_r", len(labels))
    legend_handles = [
        Patch(facecolor=cmap(i / (len(labels) - 1)), edgecolor="black", linewidth=0.5, label=labels[i])
        for i in range(len(labels))
    ]
    legend_handles.append(Patch(facecolor="grey", edgecolor="black", linewidth=0.5, label="No observations"))

    map_df.plot(
        column="year_bin",
        cmap="Blues_r",
        linewidth=0.5,
        edgecolor="black",
        missing_kwds={"color": "grey"},
        categorical=True,
        legend=False,
        ax=ax
    )

    ax.legend(handles=legend_handles, title="Year", title_fontsize=12, loc="lower left")
    ax.set_title(f"Countries with Observations up to (including) {month-1}", fontsize=15)
    ax.set_axis_off()
    
    filepath = f"../Products/Monthly_first_obs_Maps/{month-1}.png"
    plt.savefig(filepath, dpi=300, bbox_inches="tight")
    plt.close()

    saved_files.append(filepath) # for the following GIF

In [ ]:
# create GIF

own.gif_maker(*saved_files, path="C:/Studium/X_Masterarbeit/Data/Products/Monthly_first_obs_Maps/Animation", duration=200)

Monthly Maps with active/inactive countries.

In [ ]:
world = gpd.read_file("../ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp", engine="fiona")
world = world.rename(columns={"NAME": "Country"})

df = pd.read_csv("../CWData_clean7.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])
df["year_month"] = pd.to_datetime(df["year_month"]).dt.to_period("M")

df["ISO_A3"] = df["Country"].apply(own.get_iso3)

pd.set_option("display.max_columns", None)
df.head(3)

countries_ever = set(df["ISO_A3"].dropna().unique())
all_months = pd.period_range(
    df["year_month"].min(),
    df["year_month"].max(),
    freq="M"
)

for month in all_months:
    
    # countries with observations in this month
    current_active = set(
        df.loc[df["year_month"] == month, "ISO_A3"].dropna().unique()
    )
    
    # copy of world map
    map_df = world.copy().to_crs("+proj=robin")
    
    # create status column
    def classify_country(iso):
        if iso in current_active:
            return "Active"
        elif iso in countries_ever:
            return "Inactive"
        else:
            return "Never active"
    
    map_df["status"] = map_df["ADM0_A3"].apply(classify_country)
    
    # plot
    fig, ax = plt.subplots(1, 1, figsize=(16, 8))
    
    categories = ["Active", "Inactive", "Never active"]
    map_df["status"] = pd.Categorical(map_df["status"], categories=categories)

    # define colors
    cmap = mcolors.ListedColormap(["blue", "#df6a91", "grey"])
    color_map = {
        "Active": "blue",
        "Inactive": "#df6a91",
        "Never active": "grey"
    }

    fig, ax = plt.subplots(1, 1, figsize=(16, 8))

    map_df.plot(
        column="status",
        categorical=True,
        cmap=cmap,
        linewidth=0.5,
        edgecolor="black",
        legend=False,
        ax=ax
    )

    legend_handles = [
        Patch(facecolor=color, edgecolor="black", linewidth=0.5, label=label)
        for label, color in color_map.items()
    ]
    
    ax.legend(handles=legend_handles, title="Country Status", title_fontsize=12, loc="lower left")
    ax.set_title(f"Active Countries in {month}", fontsize=15)
    ax.set_axis_off()
    
    plt.savefig(f"../Products/Monthly_active_Countries_Maps/{month}.png",
                dpi=300, bbox_inches="tight")
    plt.close()

Now do the same thing for actual spots, not whole countries.

In [ ]:
world = gpd.read_file("../ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp", engine="fiona")
world = world.rename(columns={"NAME": "Country"})

df = pd.read_csv("../CWData_clean7.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])
df["year_month"] = pd.to_datetime(df["year_month"]).dt.to_period("M")

df["geometry"] = df.apply(
    lambda row: Point(row["longitude"], row["latitude"]), axis=1
)

gdf = gpd.GeoDataFrame(
    df,
    geometry="geometry",
    crs="EPSG:4326"
).to_crs("+proj=robin")

all_months = pd.period_range(
    gdf["year_month"].min(),
    gdf["year_month"].max(),
    freq="M"
)

for month in all_months:

    fig, ax = plt.subplots(1, 1, figsize=(16, 8))
    
    # map below
    world.to_crs("+proj=robin").plot(
        ax=ax,
        color="grey",
        edgecolor="black",
        linewidth=0.5
    )
    
    # active spots in this month
    active = gdf[gdf["year_month"] == month]
    
    # all other spots
    inactive = gdf[gdf["year_month"] < month]
    
    # first plot inactive ones
    inactive.plot(
        ax=ax,
        color="red",
        markersize=5,
        alpha=0.4
    )
    
    # then active ones
    active.plot(
        ax=ax,
        color="blue",
        markersize=8,
        alpha=0.8
    )

    active_patch = mpatches.Patch(facecolor="blue", edgecolor="black", linewidth=0.5, label="Active")
    inactive_patch = mpatches.Patch(facecolor="red", edgecolor="black", linewidth=0.5, label="Inactive")

    legend = ax.legend(
        handles=[active_patch, inactive_patch],
        title="Spot Status",
        loc="lower left",
        frameon=True
    )

    legend.get_title().set_fontsize(12)
    for text in legend.get_texts():
        text.set_fontsize(10)
    
    ax.set_title(f"Active Measurement Spots in {month}", fontsize=16)
    ax.set_axis_off()
    
    plt.savefig(f"../Products/Monthly_active_Spots_Maps/{month}.png",
                dpi=300, bbox_inches="tight")
    plt.close()

Monthly Maps of the number of active users per country

In [ ]:
world = gpd.read_file("../ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp", engine="fiona")
world = world.rename(columns={"NAME": "Country"})

df = pd.read_csv("../CWData_clean7.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])
df["year_month"] = pd.to_datetime(df["year_month"]).dt.to_period("M")
df["ISO_A3"] = df["Country"].apply(own.get_iso3)

countries_ever = set(df["ISO_A3"].dropna().unique())
all_months = pd.period_range(
    df["year_month"].min(),
    df["year_month"].max(),
    freq="M"
)

# define bins
bins = [0, 1, 5, 10, 20, 50, float("inf")]
labels = ["0", "1-5", "6-10", "11-20", "21-50", ">51"]
colors = ["#ffffff", "#9ecae1", "#6baed6", "#3182bd", "#08519c", "#08306b"]
cmap = mcolors.ListedColormap(colors)
norm = mcolors.BoundaryNorm(bins, cmap.N)

legend_handles = [
    Patch(facecolor="grey", edgecolor="black", linewidth=0.5, label="Never active"),
    Patch(facecolor="white", edgecolor="black", linewidth=0.5, label="No observations (this month)"),
] + [
    Patch(facecolor=colors[i+1], edgecolor="black", linewidth=0.5, label=labels[i+1])
    for i in range(len(labels)-1)
]

for month in all_months:

    user_counts = (
        df[df["year_month"] == month]
        .groupby("ISO_A3")["spotted_by"]
        .nunique()
    )

    map_df = world.copy().to_crs("+proj=robin")
    map_df["user_count"] = map_df["ADM0_A3"].map(user_counts)

    def assign_value(row):
        if row["ADM0_A3"] not in countries_ever:
            return -1
        elif pd.isna(row["user_count"]):
            return 0
        else:
            return row["user_count"]

    map_df["plot_value"] = map_df.apply(assign_value, axis=1)

    fig, ax = plt.subplots(1, 1, figsize=(16, 8))

    # plot never active countries grey
    map_df[map_df["plot_value"] == -1].plot(
        ax=ax, color="grey", edgecolor="black", linewidth=0.5, zorder=1
    )

    # plot currently inactive countries white
    map_df[map_df["plot_value"] == 0].plot(
        ax=ax, color="white", edgecolor="black", linewidth=0.5, zorder=1
    )

    # plot active countries according to previously specified colors
    map_df[map_df["plot_value"] > 0].plot(
        column="plot_value",
        ax=ax,
        cmap=cmap,
        norm=norm,
        edgecolor="black",
        linewidth=0.5,
        zorder=2,
        legend=False
    )

    ax.legend(
        handles=legend_handles,
        title="Active Users",
        title_fontsize=11,
        fontsize=9,
        loc="lower left"
    )
    ax.set_title(f"Active Users per Country in {month}", fontsize=15)
    ax.set_axis_off()

    plt.savefig(f"../Products/Monthly_active_Users_Maps/{month}.png",
                dpi=300, bbox_inches="tight")
    plt.close()

Now we generate a dataframe with monthly percentages of hydrological variables measured. E.g. for Switzerland, in January 2024, 32% virtual scale, 17% physical scale, 5% soil moisture, 40% plastic pollution, 1% temporary stream, 3% stream type and 2% standing water type MIGHT be observed.

In [18]:
df = pd.read_csv("../CWData_clean7.csv")
df["year_month"] = pd.to_datetime(df["year_month"]).dt.to_period("M")

category_cols = [
    "physical scale",
    "plastic pollution",
    "soil moisture",
    "standing water type",
    "stream type",
    "temporary stream",
    "virtual scale"
]

counts = (
    df
    .groupby(["Country", "year_month", "Category"])
    .size()
    .reset_index(name="n_obs")
)

totals = (
    df
    .groupby(["Country", "year_month"])
    .size()
    .reset_index(name="total_obs")
)

counts = counts.merge(totals, on=["Country","year_month"])
counts["percent"] = round(counts["n_obs"] / counts["total_obs"] * 100,2)

category_percentages = (
    counts
    .pivot_table(
        index=["Country","year_month"],
        columns="Category",
        values="percent",
        fill_value=0
    )
    .reset_index()
)

#category_percentages["year_month"] = category_percentages["year_month"].dt.to_period("M")
full_range = pd.period_range("2017-02", "2026-01", freq="M")
countries = category_percentages["Country"].unique()
full_index = pd.MultiIndex.from_product(
    [countries, full_range],
    names=["Country", "year_month"]
)

category_percentages_full = (
    category_percentages
    .set_index(["Country","year_month"])
    .reindex(full_index)
    .reset_index()
)

category_percentages_full[category_cols] = category_percentages_full[category_cols].fillna(0)
mask = category_percentages_full[category_cols].any(axis=1)
category_percentages_full["top_category"] = None
category_percentages_full.loc[mask, "top_category"] = (
    category_percentages_full.loc[mask, category_cols]
    .idxmax(axis=1)
)

category_percentages_full["ISO_A3"] = category_percentages_full["Country"].apply(own.get_iso3)

C:\Users\yanni\AppData\Local\Temp\ipykernel_3356\2214125278.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 4

In [22]:
category_percentages_full.head()

Category,Country,year_month,physical scale,plastic pollution,soil moisture,standing water type,stream type,temporary stream,virtual scale,top_category,ISO_A3
0,Afghanistan,2017-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,AFG
1,Afghanistan,2017-03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,AFG
2,Afghanistan,2017-04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,AFG
3,Afghanistan,2017-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,AFG
4,Afghanistan,2017-06,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,AFG


Now create monthly maps of the top category

In [ ]:
# uses category_percentages_full

world = gpd.read_file(f"../ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp", engine="fiona").to_crs("+proj=robin")
countries_with_obs = category_percentages_full["ISO_A3"].unique()

category_colors = {
    "physical scale": "#1418fc",
    "plastic pollution": "#fbff2b",
    "soil moisture": "#82571b",
    "standing water type": "#dd6ef0",
    "stream type": "#fc1471",
    "temporary stream": "#8c14fc",
    "virtual scale": "#14c2fc"
}

months = category_percentages_full["year_month"].unique()

for month in months:

    month_df = category_percentages_full[
        category_percentages_full["year_month"] == month
    ]
    
    map_df = world.merge(
        month_df,
        left_on="ADM0_A3",
        right_on="ISO_A3",
        how="left"
    )

    fig, ax = plt.subplots(1,1, figsize=(16,8))

    # Basemap --> countries without any observations ever
    world.plot(
        ax=ax,
        color="grey",
        edgecolor="black",
        linewidth=0.5
    )

    # countries without observations in this specific month
    no_obs_this_month = map_df[
        (map_df["ADM0_A3"].isin(countries_with_obs)) &
        (map_df["top_category"].isna())
    ]

    if not no_obs_this_month.empty:
        no_obs_this_month.plot(
            ax=ax,
            color="lightgrey",
            edgecolor="black",
            linewidth=0.5
        )

    #countries with observations in this specific month
    for category, color in category_colors.items():
        subset = map_df[map_df["top_category"] == category]
        if not subset.empty:
            subset.plot(
                ax=ax,
                color=color,
                edgecolor="black",
                linewidth=0.5
            )
        
    legend_order = [
        "physical scale",
        "virtual scale",
        "soil moisture",
        "stream type",
        "temporary stream",
        "plastic pollution",
        "standing water type",
        "No observations this month",
        "No observations"
    ]
    legend_handles = [
        Patch(facecolor=category_colors.get(label, "lightgrey" if label == "No observations this month" else "grey"), label=label, edgecolor="black", linewidth=0.5)
        for label in legend_order
    ]
    ax.legend(handles=legend_handles, loc="lower left", title="Category", title_fontsize=12)
    
    ax.axis("off")
    plt.title(f"Dominant CrowdWater Category Type - {month}", fontsize=15)
    plt.savefig(f"../Products/Hydro_Categories/top_category/top_category_{month}.png", dpi=300, bbox_inches="tight")
    plt.close()

And now monthly maps for each category that shows the percentage of observations per country

In [ ]:
def category_plotter(df, category):
    world = gpd.read_file(f"../ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp", engine="fiona").to_crs("+proj=robin")
    countries_with_obs = df["ISO_A3"].unique()
    months = df["year_month"].unique()

    bins = [0, 20, 40, 60, 80, 100]
    labels = ["0-20%", ">20-40%", ">40-60%", ">60-80%", ">80-100%"]
    bin_col = category + "_binned"

    cmap = get_cmap("Blues", len(labels))
    legend_handles = [
        Patch(facecolor=cmap(i / (len(labels) - 1)), label=labels[i], edgecolor="black", linewidth=0.5, )
        for i in range(len(labels))
    ]
    
    legend_handles.append(Patch(facecolor="lightgrey", edgecolor="black", linewidth=0.5, label="No observations this month"))
    legend_handles.append(Patch(facecolor="grey", edgecolor="black", linewidth=0.5, label="No observations"))
    
    for month in months:
        month_df = df[df["year_month"] == month].copy()
        month_df[bin_col] = pd.cut(month_df[category], bins=bins, labels=labels, include_lowest=True)
        month_df["_codes"] = month_df[bin_col].cat.codes
        
        map_df = world.merge(
            month_df,
            left_on="ADM0_A3",
            right_on="ISO_A3",
            how="left"
        )

        fig, ax = plt.subplots(1, 1, figsize=(16, 8))
        world.plot(ax=ax, color="grey", edgecolor="black", linewidth=0.5)
        
        map_df[map_df["ADM0_A3"].isin(countries_with_obs)].plot(
            ax=ax, color="lightgrey", edgecolor="black", linewidth=0.5
        )

        subset = map_df[map_df[category] > 0].copy()
        if not subset.empty:
            subset.plot(
                column="_codes", cmap="Blues", legend=False,
                vmin=0, vmax=len(labels) - 1,
                ax=ax, edgecolor="black", linewidth=0.5
            )

        ax.legend(handles=legend_handles, loc="lower left", title=f"% of observations", title_fontsize=12)
        ax.axis("off")
        plt.title(f"Share of Observations in Category {category.title()} - {month}", fontsize=15)
        plt.savefig(f"../Products/Hydro_Categories/{category.replace(' ', '_')}/{category.replace(' ', '_')}_{month}.png", dpi=300, bbox_inches="tight")
        plt.close()

In [ ]:
# uses category_percentages_full

categories = ["physical scale",
            "virtual scale",
            "soil moisture",
            "stream type",
            "temporary stream",
            "plastic pollution",
            "standing water type"
        ]
for category in categories:
    category_plotter(category_percentages_full, category)

And also create a graph that shows average percentages for each country, yearly

In [ ]:
# uses category_percentages_full

cat_cols = ["physical scale", "virtual scale", "soil moisture", "stream type", "temporary stream", "plastic pollution", "standing water type"]

country_totals = category_percentages_full.groupby("Country")[cat_cols].sum() # sum per country

country_pct = country_totals.div(country_totals.sum(axis=1), axis=0) * 100 # percentages per country

country_pct = country_pct.sort_values("Country", ascending=False) # sort alphabetically

colors = ["#1418fc", "#14c2fc", "#82571b", "#fc1471", "#8c14fc", "#fbff2b", "#dd6ef0"]
color_dict = dict(zip(cat_cols, colors))

fig, ax = plt.subplots(figsize=(12, len(country_pct) * 0.4))

lefts = pd.Series([0.0] * len(country_pct), index=country_pct.index)

for cat, color in color_dict.items():
    vals = country_pct[cat]
    ax.barh(
        country_pct.index,
        vals,
        left=lefts,
        color=color,
        label=cat,
        edgecolor="none"
    )
    lefts += vals

ax.set_xlim(0, 100)
ax.set_ylim(-0.5, len(country_pct) - 0.5)
ax.set_xlabel("Percentage of Observations (%)", fontsize=12)
ax.set_title("Observation Categories per Country", fontsize=15)
ax.grid(axis="x", linestyle="--", color="black", alpha=0.8)

patches = [mpatches.Patch(facecolor=color_dict[cat], label=cat) for cat in cat_cols]
ax.legend(handles=patches, bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=10, frameon=True)

plt.savefig("../Products/Hydro_Categories/barchart_category_per_country.png", dpi=300, bbox_inches="tight")
plt.close()